# 03 – Thí nghiệm và chốt mô hình tốt nhất

**Ý tưởng:** thay vì chọn đại một cấu hình, ta chạy nhiều cấu hình khác nhau (gọi là *thí nghiệm*), so sánh công bằng trên cùng dữ liệu, rồi chọn cấu hình tốt nhất.

**Quy trình của notebook:**
1. Chạy 5 thí nghiệm, mỗi thí nghiệm train một model.
2. So sánh các model trên tập **validation**.
3. Chọn thí nghiệm thắng (val accuracy cao nhất, hòa thì val loss thấp hơn).
4. Chấm điểm model thắng **một lần duy nhất** trên tập test.
5. Lưu model thắng vào `models/best_model.h5` để API dùng.

**Vì sao chọn bằng validation mà không dùng test?** Nếu chọn theo điểm test thì model được chọn sẽ vô tình khớp với tập test, và điểm test không còn phản ánh chất lượng thật. Giữ test làm bài kiểm tra cuối cho kết quả trung thực hơn.

**File được ghi ra:**
- `reports/experiments.csv`, `reports/experiments_comparison.png`: so sánh các thí nghiệm
- `reports/best_experiment.json`: cấu hình và điểm của thí nghiệm thắng
- `models/best_model.h5`, `models/class_names.json`: model mà API sẽ dùng

> **Cảnh báo:** chạy xong sẽ **ghi đè** `models/best_model.h5` bằng model thắng. Thời gian chạy phụ thuộc máy (có GPU nhanh hơn nhiều). Chạy hết 5 thí nghiệm trên CPU có thể mất rất lâu.

## 1. Danh sách thí nghiệm

Mỗi thí nghiệm thay đổi một vài yếu tố so với baseline (E2) để xem yếu tố đó có giúp ích không:

| Thí nghiệm | Thay đổi | Câu hỏi muốn trả lời |
|---|---|---|
| **E1** `cnn_scratch` | CNN nhỏ tự xây, không pretrained | Có cần dùng mạng pretrained không? |
| **E2** `mnv2_frozen_lr1e-3` | MobileNetV2 đóng băng, lr 1e-3 | **Baseline** (mốc so sánh) |
| **E3** `mnv2_frozen_lr1e-4` | Như E2 nhưng lr nhỏ hơn 10 lần | Học chậm hơn nhưng ổn định hơn có tốt hơn không? |
| **E4** `mnv2_finetune_last30` | E2, sau đó mở khóa 30 lớp cuối và train tiếp với lr rất nhỏ | Tinh chỉnh sâu mạng pretrained có giúp nhận ra chi tiết như vết mốc không? |
| **E5** `mnv2_strong_aug_dropout0.5` | Augmentation mạnh hơn, dropout 0.5 | Tăng độ đa dạng dữ liệu và chống học vẹt có giúp model chịu được ảnh lạ không? |

Bảng dưới in ra cấu hình của cả 5 thí nghiệm (xem ý nghĩa các cột ở notebook 02).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # để import được package src

import pandas as pd
from IPython.display import Image

from src.data_loader import REPORTS_DIR
from src.experiments import EXPERIMENTS, run_experiments

display(pd.DataFrame(EXPERIMENTS)[['name', 'arch', 'lr', 'dropout', 'aug', 'finetune_layers']])

## 2. Chạy toàn bộ thí nghiệm

`run_experiments(epochs=15)` lần lượt: đọc dữ liệu (dùng bản chia cố định trong `data/processed/`), train từng model, ghi điểm validation, rồi chọn model thắng và lưu lại. Tiến trình hiện trong log phía dưới ô; kết quả được ghi dần vào `reports/experiments.csv` nên nếu bị dừng giữa chừng vẫn giữ được các thí nghiệm đã xong.

**Cách chạy thử nhanh:**
- Giảm số epoch: `run_experiments(epochs=3)`
- Chỉ chạy một vài thí nghiệm: `run_experiments(epochs=15, names=['E1_cnn_scratch', 'E2_mnv2_frozen_lr1e-3'])`

Hàm trả về `results` (bảng so sánh) và `best` (thông tin thí nghiệm thắng).

In [ ]:
results, best = run_experiments(epochs=15)

## 3. Bảng so sánh (đã xếp từ tốt đến kém)

**Ý nghĩa các cột:**

| Cột | Ý nghĩa | Tốt khi |
|---|---|---|
| `val_acc` | Độ chính xác trên tập validation (tiêu chí chọn chính) | Càng cao càng tốt |
| `val_loss` | Độ sai lệch trên validation (dùng để phân thắng bại khi `val_acc` bằng nhau) | Càng thấp càng tốt |
| `train_acc_last` | Độ chính xác trên tập train ở epoch cuối | Gần với `val_acc` |
| `epochs_run` | Số epoch thực tế đã chạy (EarlyStopping có thể dừng sớm) | Chỉ để tham khảo |
| `minutes` | Thời gian train | Càng ít càng tiết kiệm |

**Cách đọc:**
- Hàng đầu tiên là thí nghiệm thắng.
- Nếu `train_acc_last` cao hơn `val_acc` nhiều (ví dụ 99% so với 90%), model đang học vẹt. Khi hai thí nghiệm có `val_acc` gần nhau, nên ưu tiên cái có khoảng cách này nhỏ hơn và ít tốn thời gian hơn.
- Chênh lệch `val_acc` vài phần mười % giữa các thí nghiệm có thể chỉ do ngẫu nhiên, đừng coi đó là khác biệt thật.

Biểu đồ cột bên dưới là hình ảnh trực quan của cùng bảng này.

In [ ]:
display(results[['name', 'val_acc', 'val_loss', 'train_acc_last', 'epochs_run', 'minutes']])
Image(os.path.join(REPORTS_DIR, 'experiments_comparison.png'))

## 4. Thí nghiệm được chốt

In ra nội dung `reports/best_experiment.json` gồm:
- `selection_rule`: quy tắc đã dùng để chọn.
- `config`: cấu hình đầy đủ của thí nghiệm thắng (dùng để huấn luyện lại bằng `python -m src.train`).
- `val`: điểm trên tập validation (dùng để chọn).
- `test`: điểm trên tập test (chỉ dùng báo cáo cuối, chấm một lần).

Ảnh phía dưới là **confusion matrix** của model thắng trên tập test (cách đọc xem notebook 02). Hai ô ngoài đường chéo là các ảnh bị đoán sai; hãy xem model hay nhầm theo hướng nào: coi `rotten` là `fresh` (bỏ sót trái hỏng) hay ngược lại (loại nhầm trái tốt).

In [ ]:
import json
print(json.dumps(best, ensure_ascii=False, indent=2))
Image(os.path.join(REPORTS_DIR, 'confusion_matrix.png'))

## 5. Giới hạn cần biết và bước tiếp theo

**Giới hạn:**
- Tập validation và tập test có cùng kiểu ảnh với tập train (ảnh một quả, nền đơn giản). Điểm cao ở đây chưa đảm bảo model đúng với ảnh ngoài đời (tay cầm, nền lộn xộn, nhiều quả trong một ảnh).
- Mỗi thí nghiệm chỉ chạy một lần với một hạt giống ngẫu nhiên, nên khác biệt nhỏ giữa các thí nghiệm có thể do may rủi.
- Model chỉ có hai nhãn nên với ảnh không phải trái cây vẫn sẽ trả về `fresh` hoặc `rotten`.

**Bước tiếp theo:**
1. Tắt và chạy lại API để nạp model mới: `uvicorn api.main:app --reload --port 8000`, rồi thử bằng Postman với cả ảnh trong dataset lẫn ảnh thực tế.
2. Nếu model sai với ảnh thực tế, thêm ảnh thật vào `data/raw/train/` (đúng thư mục nhãn), chạy `python -m src.data_loader --refresh`, rồi chạy lại notebook này.
3. Muốn huấn luyện lại đầy đủ (30 epoch) bằng cấu hình thắng: `python -m src.train`.